# 02 Clean identifier links

目标：分别清洗CRSP share-class静态映射与S12日期映射，排除一对多键。

- 输入：crsp_fundno_wficn.csv；fundno_wficn.csv
- 输出：两张clean Parquet及两张冲突审计表
- 主键：crsp_fundno；fundno+fdate+rdate
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：01
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import polars as pl
from datetime import date
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.runtime_paths import output_path
from src.schema_utils import assert_unique

n_rows = None if not SMOKE else 100_000
crsp_raw = pl.read_csv(PATHS["mflink_crsp"], columns=["crsp_fundno", "wficn"], n_rows=n_rows, infer_schema=False)
s12_raw = pl.read_csv(PATHS["mflink_s12"], columns=["fundno", "fdate", "rdate", "wficn", "assets"], n_rows=n_rows, infer_schema=False)

crsp = crsp_raw.unique(maintain_order=True).with_columns(
    pl.col("wficn").str.strip_chars().replace("", None)
).filter(pl.col("wficn").is_not_null())
crsp_conflict_keys = crsp.group_by("crsp_fundno").agg(pl.col("wficn").n_unique().alias("n_wficn")).filter(pl.col("n_wficn") > 1)
crsp_conflicts = crsp.join(crsp_conflict_keys, on="crsp_fundno", how="inner").with_columns(pl.lit("multiple_wficn_for_crsp_fundno").alias("reason_code"))
crsp_clean = crsp.join(crsp_conflict_keys.select("crsp_fundno"), on="crsp_fundno", how="anti")
assert_unique(crsp_clean, ["crsp_fundno"], "clean CRSP link")

s12 = s12_raw.rename({"assets": "mflink_assets"}).unique(maintain_order=True).with_columns(
    pl.col("wficn").str.strip_chars().replace("", None),
    pl.col("fdate").str.to_date(strict=False),
    pl.col("rdate").str.to_date(strict=False),
    pl.col("mflink_assets").cast(pl.Float64, strict=False),
)
s12 = s12.filter(pl.col("wficn").is_not_null())
s12_conflict_keys = s12.group_by(["fundno", "fdate", "rdate"]).agg(pl.col("wficn").n_unique().alias("n_wficn")).filter(pl.col("n_wficn") > 1)
s12_conflicts = s12.join(s12_conflict_keys, on=["fundno", "fdate", "rdate"], how="inner").with_columns(pl.lit("multiple_wficn_for_s12_key").alias("reason_code"))
s12_clean = s12.join(s12_conflict_keys.select(["fundno", "fdate", "rdate"]), on=["fundno", "fdate", "rdate"], how="anti")
assert_unique(s12_clean, ["fundno", "fdate", "rdate"], "clean S12 link")
assert "assets" not in s12_clean.columns and "mflink_assets" in s12_clean.columns

crsp_out = output_path(ROOT, RUN_MODE, "02_clean_identifier_links", "data/interim/02_links/crsp_fundno_wficn_clean.parquet")
s12_out = output_path(ROOT, RUN_MODE, "02_clean_identifier_links", "data/interim/02_links/fundno_wficn_clean.parquet")
audit1 = output_path(ROOT, RUN_MODE, "02_clean_identifier_links", "data/audit/02_crsp_link_conflicts.csv")
audit2 = output_path(ROOT, RUN_MODE, "02_clean_identifier_links", "data/audit/02_s12_link_conflicts.csv")
for frame, path, writer in [(crsp_clean, crsp_out, write_parquet_atomic), (s12_clean, s12_out, write_parquet_atomic), (crsp_conflicts, audit1, write_csv_atomic), (s12_conflicts, audit2, write_csv_atomic)]:
    writer(frame, path)

INPUTS_USED=["Data_processing/manual_processing/crsp_fundno_wficn.csv","Data_processing/manual_processing/fundno_wficn.csv"]
OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [crsp_out,s12_out,audit1,audit2]]
INPUT_COUNTS={"crsp_link":crsp_raw.height,"s12_link":s12_raw.height}
OUTPUT_COUNTS={"crsp_clean":crsp_clean.height,"s12_clean":s12_clean.height}
PRIMARY_KEY=["crsp_fundno"]
DUPLICATE_KEY_COUNT=0
EXCLUDED_ROW_COUNT=crsp_conflicts.height+s12_conflicts.height
EXCEPTION_FILE=str(audit1.relative_to(ROOT))
STAGE_STATUS="passed"; STAGE_NOTES=["Two mapping systems were kept separate.","mflink assets renamed to mflink_assets."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "02_clean_identifier_links", stage="02", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
